# 052 — Set up the FEMA P695 IDA runs, one per design group

The `ida_femap695` arm of the WP1 fragility comparison is a hunt-trace-fill IDA over the
22 FEMA P695 far-field records. Because the record set is fixed, **the analysis depends only
on the structure** — exactly the property notebook `054` exploits for MSA-FEMAP695.

Notebook `051` showed the 120 (site, storey) designs collapse to **51 unique designs**
(25 × 3s, 26 × 5s), so this notebook sets the IDA up **per design group**, not per site.
Notebook `011` still writes a per-site IDA (60 per storey); it is left alone, but for the 5s
structures — whose IDAs have not been run — the group runs replace it, 26 analyses instead
of 60.

## What this notebook writes

1. **Analysis folders** under `DEST_ROOT/group_{n}s_{gid}/{n}s/mdof/`
   (`DEST_ROOT = analysis_data.wp1_fixed_record_sets`, i.e. `D:/08_wp1_fixed_record_sets`) —
   the same folders `054` uses, so a group ends up with its structural model, a modal
   analysis, the FEMA P695 IDA **and** the MSA-FEMAP695 stripes side by side. The `{n}s`
   level is redundant with the group name but keeps the folder depth identical to the
   site-specific `site_{ii}/{n}s/mdof/` tree, so the same launchers, `window_name_index` and
   post-processing paths work for both. No cyclic pushover, no MSA files.
2. **Migrated results** — where the group's representative site already has
   `ida_femap695` output under `SITE_ROOT/site_{ii}/{n}s/mdof/` (true for all 3s sites),
   that folder is *copied* into the group folder and verified, so the group folder becomes
   the single home of the result. Nothing is deleted (see §4).
3. **Batch launchers** at
   `phd_project/scripts/WP1_ground_motion_set/batch_run_analyses/fixed_record_sets/{n}s/mdof/ida_femap695.py`,
   listing only the groups that still need running.
4. **A group → site map** at `results/06_group_ida_femap695/group_folder_map.csv`, so the
   fragility fitted from one group's IDA can be fanned back out to its member sites.

## Where this sits in the pipeline

```
011  per-site designs                 051  design groups
              \                        /
               052 (this)  ->  run the IDAs  ->  053 convert fragilities to AvgSA
                                                        |
                                                        v
                                          054  MSA-FEMAP695 stripes (needs the fragility)
```

`054` inverts each group's `ida_femap695` AvgSA_03 fragility to place its stripes, which is
why every 5s group is skipped there today. This notebook is the step that unblocks it.

## Dependencies

- **011** — the per-site designs (`{tag}_out.json`) the group models are copied from, and
  the existing per-site `ida_femap695` results that §3 migrates.
- **051** — `unique_structural_designs.csv`, the design groups.

## Running this notebook without the analysis drive

Set `BUILD_ANALYSIS_FOLDERS = False` to write only the in-repo artefacts (launchers, group
map) and skip both the folder creation on `D:` and the migration. Everything else, including
the paths written into the launchers, is unaffected.

In [1]:
# %load_ext autoreload
# %autoreload 2

## 0. Setup & parameters

In [2]:
import shutil
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_n_damping_modes_from_design_file,
    get_n_primary_modes_from_design_file,
)
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_analysis_config,
    copy_batch_ida_buildings,
    copy_file,
    copy_nlcbf_model,
)
from phd_project.scripts.WP1_ground_motion_set.design_groups import load_design_groups

cfg = config.load_config()

In [3]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Root of the (large) analysis folders. Written as DEST_ROOT/group_{n}s_{gid}/{n}s/mdof/ -
# the same tree notebook 054 writes the MSA-FEMAP695 runs into.
DEST_ROOT = Path(cfg["analysis_data"]["wp1_fixed_record_sets"])

# Per-site analysis folders from nb 011 - the source of the results migrated in section 3.
SITE_ROOT = Path(cfg["analysis_data"]["wp1_casestudy_sites"])

# The site-specific designs from nb 011, copied into each group folder.
DESIGN_ROOT = Path(cfg["models"]["casestudy_designs_site_specific"])

# The DEST_ROOT folders can only be written where the analysis drive is attached. False
# writes just the in-repo artefacts - the launchers and the group map - and the launcher
# paths are the DEST_ROOT ones either way. SET THIS TO True ON THE ANALYSIS MACHINE.
BUILD_ANALYSIS_FOLDERS = False

# Copy each representative's existing ida_femap695 results into its group folder (3.2).
MIGRATE_EXISTING_IDA = True

# Stage 2 of the optional cleanup in section 4. DESTRUCTIVE - read that section first.
DELETE_SITE_IDA = False

# --- groups ---
STOREYS = [3, 5]

# --- IDA (FEMA P695 far-field set), all matching nb 011 ---
IDA_TAG = "ida_femap695"                        # results folder name
CONFIG_NAME = "config_ida_htf_femap695.py"
GM_JSON_SRC = "D:/gm_records_p695"              # records as seen by the analysis machine
DO_FILL = False
FEMAP695_RECORDS = [
    "fema_p695_120111.json", "fema_p695_120121.json", "fema_p695_120411.json",
    "fema_p695_120521.json", "fema_p695_120611.json", "fema_p695_120621.json",
    "fema_p695_120711.json", "fema_p695_120721.json", "fema_p695_120811.json",
    "fema_p695_120821.json", "fema_p695_120911.json", "fema_p695_120921.json",
    "fema_p695_121011.json", "fema_p695_121021.json", "fema_p695_121111.json",
    "fema_p695_121211.json", "fema_p695_121221.json", "fema_p695_121321.json",
    "fema_p695_121411.json", "fema_p695_121421.json", "fema_p695_121511.json",
    "fema_p695_121711.json",
]

# --- structural model (matches nb 011 / 054) ---
DAMPING_RATIO = 0.05
MDOF_DRIFT_LIMIT = 0.2                          # MDOF collapse drift limit
RECORDER_KEY = "ida_process_recorder_roof&maxstorey_drift"

# --- batch launchers ---
# False lists only the groups with no results yet; True lists every group.
LIST_ALL_GROUPS = False
BATCH_BASE = Path(cfg["scripts"]["batch_run_analyses"]) / "fixed_record_sets"
BATCH_NAME = "ida_femap695.py"

# --- group -> site map ---
MAP_DST = Path(cfg["results"]["group_ida_femap695"])
MAP_DST.mkdir(parents=True, exist_ok=True)

print(f"DEST_ROOT  = {DEST_ROOT}  (build folders: {BUILD_ANALYSIS_FOLDERS})")
print(f"SITE_ROOT  = {SITE_ROOT}  (migrate results: {MIGRATE_EXISTING_IDA})")
print(f"BATCH_BASE = {BATCH_BASE}")
print(f"MAP_DST    = {MAP_DST}")

DEST_ROOT  = D:\08_wp1_fixed_record_sets  (build folders: False)
SITE_ROOT  = D:\07_wp1_casestudy_sites  (migrate results: True)
BATCH_BASE = C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\batch_run_analyses\fixed_record_sets
MAP_DST    = C:\Users\clemettn\Documents\phd\results\06_group_ida_femap695


## 1. The building groups

One row per design group from notebook 051 (`is_representative == True`). `group_id`
restarts at 0 for each storey count, so the folder name carries both:
`group_{n}s_{gid:02d}`.

Unlike `054` **no group is skipped** — the IDA needs no hazard, no stripes and no fragility,
only the structure.

In [4]:
groups_df = load_design_groups(cfg)

groups = (groups_df[groups_df["is_representative"] & groups_df["storeys"].isin(STOREYS)]
          .loc[:, ["storeys", "group_id", "representative_site",
                   "representative_tag", "n_sites_in_group"]]
          .sort_values(["storeys", "group_id"])
          .reset_index(drop=True))

# member sites of each group, for the map written in section 6
members = (groups_df.groupby(["storeys", "group_id"])["site"]
           .apply(lambda s: sorted(int(x) for x in s)))


def group_folder_name(n: int, gid: int) -> str:
    """Folder for a design group: group_3s_00, group_5s_07, ..."""
    return f"group_{n}s_{int(gid):02d}"


def mdof_folder(n: int, gid: int) -> Path:
    """DEST_ROOT/group_{n}s_{gid}/{n}s/mdof - the {n}s level is deliberately
    redundant, so the folder depth matches the site-specific site_{ii}/{n}s/mdof tree."""
    return DEST_ROOT / group_folder_name(n, gid) / f"{n}s" / "mdof"


def site_mdof_folder(site: int, n: int) -> Path:
    """The per-site analysis folder written by nb 011."""
    return SITE_ROOT / f"site_{int(site)}" / f"{n}s" / "mdof"


for n in STOREYS:
    n_groups = int((groups["storeys"] == n).sum())
    n_sites = int(groups_df[groups_df["storeys"] == n]["site"].nunique())
    print(f"{n}s: {n_groups} design groups covering {n_sites} sites")
groups.head()

3s: 25 design groups covering 60 sites
5s: 26 design groups covering 60 sites


,storeys,group_id,representative_site,representative_tag,n_sites_in_group
0,3,0,0,3s_cbf_dc2_site0,8
1,3,1,1,3s_cbf_dc2_site1,6
2,3,2,3,3s_cbf_dc2_site3,1
3,3,3,7,3s_cbf_dc2_site7,5
4,3,4,9,3s_cbf_dc2_site9,3


## 2. Build the group analysis folders

Each group's model is the design of its representative site, copied from
`casestudy_designs_site_specific`. As in notebook 011 two model variants are written:
`initialise_model.py` (full recorders, used by the modal analysis) and
`initialise_model_idamsa.py` (reduced recorders, used by the IDA).

The model, modal and injection files are byte-identical to the ones `054` writes, so a
folder already populated by that notebook is simply refreshed — the two notebooks can be run
in either order.

In [5]:
def build_group_ida_folder(g: dict) -> Path:
    """Write one group's mdof folder: model + modal + FEMA P695 IDA. Returns the folder."""
    n, gid, tag = g["storeys"], g["group_id"], g["representative_tag"]
    folder = mdof_folder(n, gid)
    folder.mkdir(parents=True, exist_ok=True)

    # --- design file (the model reads it from its own folder) ---
    design_src = DESIGN_ROOT / tag / f"{tag}_out.json"
    design_dst = folder / f"{tag}_designfile.json"
    copy_file(design_src, design_dst)

    # --- structural model, full recorders (modal) and reduced recorders (IDA) ---
    model_kwargs = dict(
        design_json=design_dst.name,
        damping_updates={"n_modes": get_n_damping_modes_from_design_file(design_dst),
                         "damping_ratio": DAMPING_RATIO},
        recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
    )
    init_fn_full = copy_nlcbf_model(cfg["templates"], folder, **model_kwargs)
    init_fn_reduced = copy_nlcbf_model(cfg["templates"], folder, reduced=True, **model_kwargs)

    # --- modal analysis (fundamental period) ---
    copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
    copy_analysis_config(
        cfg["templates"]["config_modal"],
        folder / "config_modal.py",
        results_folder_name="modal",
        model_file_name=init_fn_full,
        update_config={"n_modes": get_n_primary_modes_from_design_file(design_dst)},
    )

    # --- IDA coordinator / worker / helpers (mirrors add_ida_files in nb 011) ---
    copy_file(cfg["templates"][RECORDER_KEY], folder / "ida_process_recorders.py")
    copy_file(cfg["templates"]["config_im_SA"], folder / "config_im_SA.py")
    copy_file(cfg["templates"]["nltha_injection_update_damping"],
              folder / "injection_functions.py")
    copy_file(cfg["templates"]["run_batch_ida_per_record"],
              folder / "run_batch_ida_per_record.py")
    copy_file(cfg["templates"]["run_ida_htf_per_record"],
              folder / "run_ida_htf_per_record.py")

    # --- IDA config. The IM is SA(T1), built from the model by config_im_SA.py; the
    # worker resolves each record as gm_json_src / filename.
    copy_analysis_config(
        cfg["templates"]["config_ida_htf"],
        folder / CONFIG_NAME,
        results_folder_name=IDA_TAG,
        model_file_name=init_fn_reduced,
        gm_json_src_str=GM_JSON_SRC,
        record_filenames=FEMAP695_RECORDS,
        do_fill=DO_FILL,
    )
    return folder

In [6]:
built: dict[int, list[dict]] = {n: [] for n in STOREYS}

for row in groups.itertuples(index=False):
    n, gid = int(row.storeys), int(row.group_id)
    g = {
        "storeys": n,
        "group_id": gid,
        "representative_site": int(row.representative_site),
        "representative_tag": row.representative_tag,
        "n_sites": int(row.n_sites_in_group),
        "folder": mdof_folder(n, gid),
    }
    if BUILD_ANALYSIS_FOLDERS:
        design_src = DESIGN_ROOT / g["representative_tag"] / f"{g['representative_tag']}_out.json"
        if not design_src.is_file():
            print(f"WARNING: skipping group {n}s/{gid:02d}: {design_src} not found "
                  f"(run nb 011 first)")
            continue
        g["folder"] = build_group_ida_folder(g)
    built[n].append(g)

for n in STOREYS:
    verb = "built" if BUILD_ANALYSIS_FOLDERS else "listed (folders NOT built)"
    print(f"{n}s: {len(built[n])} group folder(s) {verb}")

3s: 25 group folder(s) listed (folders NOT built)
5s: 26 group folder(s) listed (folders NOT built)


## 3. Migrate the representative's existing IDA results

Every 3s site already has a FEMA P695 IDA under
`SITE_ROOT/site_{ii}/{n}s/mdof/ida_femap695/`. Within a design group those results are the
*same analysis of the same model*, so the representative's copy is exactly what the group
folder should hold — there is nothing to re-run for the 3s groups.

The copy is deliberately conservative:

- a destination that already exists is **reported and skipped**, never merged into;
- after `copytree` the two trees are compared file-by-file (relative path → size) and a
  mismatch raises;
- **nothing in `SITE_ROOT` is modified.** Removing the now-duplicated site copies is a
  separate, opt-in step — section 4.

### 3.1 Survey (read-only)

In [7]:
def tree_files(root: Path) -> dict[str, int]:
    """{relative posix path: size in bytes} for every file under `root`."""
    return {p.relative_to(root).as_posix(): p.stat().st_size
            for p in root.rglob("*") if p.is_file()}


def tree_size_mb(files: dict[str, int]) -> float:
    return sum(files.values()) / 1024**2


def src_ida_folder(g: dict) -> Path:
    return site_mdof_folder(g["representative_site"], g["storeys"]) / IDA_TAG


def dst_ida_folder(g: dict) -> Path:
    return Path(g["folder"]) / IDA_TAG


survey_rows = []
for n in STOREYS:
    for g in built[n]:
        src, dst = src_ida_folder(g), dst_ida_folder(g)
        src_files = tree_files(src) if src.is_dir() else {}
        survey_rows.append({
            "group": group_folder_name(n, g["group_id"]),
            "rep_site": g["representative_site"],
            "src_exists": src.is_dir(),
            "src_files": len(src_files),
            "src_MB": round(tree_size_mb(src_files), 1),
            "dst_exists": dst.is_dir(),
        })

survey = pd.DataFrame(survey_rows)
if survey.empty:
    print("no groups - nothing to survey")
else:
    n_src = int(survey["src_exists"].sum())
    n_dst = int(survey["dst_exists"].sum())
    print(f"{n_src}/{len(survey)} group(s) have representative IDA results to migrate "
          f"({survey['src_MB'].sum():.1f} MB); {n_dst} destination(s) already populated")
    if not SITE_ROOT.exists():
        print(f"NOTE: {SITE_ROOT} is not reachable - the analysis drive is detached, "
              f"so nothing can be migrated in this run.")
    print(survey.to_string(index=False))

0/51 group(s) have representative IDA results to migrate (0.0 MB); 0 destination(s) already populated
NOTE: D:\07_wp1_casestudy_sites is not reachable - the analysis drive is detached, so nothing can be migrated in this run.
      group  rep_site  src_exists  src_files  src_MB  dst_exists
group_3s_00         0       False          0     0.0       False
group_3s_01         1       False          0     0.0       False
group_3s_02         3       False          0     0.0       False
group_3s_03         7       False          0     0.0       False
group_3s_04         9       False          0     0.0       False
group_3s_05        12       False          0     0.0       False
group_3s_06        13       False          0     0.0       False
group_3s_07        14       False          0     0.0       False
group_3s_08        15       False          0     0.0       False
group_3s_09        17       False          0     0.0       False
group_3s_10        30       False          0     0.0       F

### 3.2 Copy and verify

In [8]:
migrated: set[tuple[int, int]] = set()      # (storeys, group_id) verified in the group folder
already_there: list[str] = []

if not (BUILD_ANALYSIS_FOLDERS and MIGRATE_EXISTING_IDA):
    print("migration skipped (BUILD_ANALYSIS_FOLDERS and MIGRATE_EXISTING_IDA must both "
          "be True)")
else:
    for n in STOREYS:
        for g in built[n]:
            key = (n, g["group_id"])
            name = group_folder_name(*key)
            src, dst = src_ida_folder(g), dst_ida_folder(g)

            if dst.is_dir():
                already_there.append(name)
                continue
            if not src.is_dir():
                continue

            shutil.copytree(src, dst)
            src_files, dst_files = tree_files(src), tree_files(dst)
            if src_files != dst_files:
                missing = set(src_files) - set(dst_files)
                differing = {k for k in set(src_files) & set(dst_files)
                             if src_files[k] != dst_files[k]}
                raise RuntimeError(
                    f"{name}: copy of {src} -> {dst} did not verify "
                    f"({len(missing)} missing, {len(differing)} size mismatch)")
            migrated.add(key)
            print(f"[migrated] {name}: {len(src_files)} files, "
                  f"{tree_size_mb(src_files):.1f} MB from site_{g['representative_site']}")

    print(f"\n{len(migrated)} group(s) migrated; {len(already_there)} already had results "
          f"in place (left untouched): {', '.join(already_there) or '-'}")

migration skipped (BUILD_ANALYSIS_FOLDERS and MIGRATE_EXISTING_IDA must both be True)


## 4. Optional: remove the now-duplicated site-specific IDA — **default off**

Once a group folder holds the verified results, the representative site's copy under
`SITE_ROOT` is redundant. Deleting it keeps one home per result, but it **discards expensive
analysis output**, so it is deliberately two-stage: the dry run below always prints exactly
what would go, and nothing is removed unless you set `DELETE_SITE_IDA = True` and re-run the
second cell.

Scope, deliberately narrow:

- only groups **verified as migrated in this session** (`migrated`) are eligible — a
  destination that merely already existed is not enough;
- only the **representative** site is touched. Non-representative members of the group keep
  their duplicate results.

Before enabling it, note that **notebook `014` reads
`SITE_ROOT/site_{ii}/{n}s/mdof/ida_femap695`** to convert the fragilities to AvgSA. Deleting
these breaks a re-run of `014` (the fragility JSONs it already wrote are unaffected). The
group-aware replacement is `053` — see the pipeline sketch at the top.

In [9]:
# Files nb 011 wrote to drive the site-specific IDA. The results folder goes with them.
IDA_SCAFFOLDING = [
    "config_ida_htf_femap695.py",
    "run_batch_ida_per_record.py",
    "run_ida_htf_per_record.py",
    "ida_process_recorders.py",
    "config_im_SA.py",
]


def deletion_targets() -> list[dict]:
    """What stage 2 would delete: one entry per verified-migrated group."""
    targets = []
    for n in STOREYS:
        for g in built[n]:
            if (n, g["group_id"]) not in migrated:
                continue
            site_folder = site_mdof_folder(g["representative_site"], n)
            results = site_folder / IDA_TAG
            files = tree_files(results) if results.is_dir() else {}
            targets.append({
                "group": group_folder_name(n, g["group_id"]),
                "results": results,
                "scaffolding": [site_folder / f for f in IDA_SCAFFOLDING
                                if (site_folder / f).is_file()],
                "n_files": len(files),
                "MB": tree_size_mb(files),
            })
    return targets


targets = deletion_targets()
if not targets:
    print("nothing eligible for deletion (no group was verified as migrated in this run)")
else:
    for t in targets:
        print(f"[would delete] {t['group']}: {t['results']}  "
              f"({t['n_files']} files, {t['MB']:.1f} MB) "
              f"+ {len(t['scaffolding'])} scaffolding file(s)")
    print(f"\n{len(targets)} group(s), {sum(t['MB'] for t in targets):.1f} MB total. "
          f"DELETE_SITE_IDA = {DELETE_SITE_IDA}")

nothing eligible for deletion (no group was verified as migrated in this run)


In [10]:
if not DELETE_SITE_IDA:
    print("DELETE_SITE_IDA is False - nothing deleted (this is the default)")
else:
    for t in deletion_targets():
        if t["results"].is_dir():
            shutil.rmtree(t["results"])
        for f in t["scaffolding"]:
            f.unlink()
        print(f"[deleted] {t['group']}: {t['results']} + {len(t['scaffolding'])} file(s)")

DELETE_SITE_IDA is False - nothing deleted (this is the default)


## 5. Batch launchers

One `run_batch_ida_buildings` launcher per storey, at
`BATCH_BASE/{n}s/mdof/ida_femap695.py`. `window_name_index=2` puts the `group_{n}s_{gid}`
folder in the console title (the config sits at `group_.../{n}s/mdof/`, so two levels up is
the group); the template default of 1 would title every window `mdof`.

A group whose folder already holds results — migrated here or otherwise — is **left out of
the launcher**, so running it cannot redo work that was just copied in. Set
`LIST_ALL_GROUPS = True` to list every group regardless.

The paths written are always the `DEST_ROOT` ones, whether or not the folders were created
here — the launcher is run on the analysis machine.

In [11]:
def has_results(g: dict) -> bool:
    """True when the group folder already holds IDA output."""
    return (g["storeys"], g["group_id"]) in migrated or dst_ida_folder(g).is_dir()


needs_run: dict[int, list[dict]] = {n: [] for n in STOREYS}

for n in STOREYS:
    needs_run[n] = [g for g in built[n] if LIST_ALL_GROUPS or not has_results(g)]
    if not needs_run[n]:
        print(f"{n}s: every group already has results - no launcher written")
        continue

    buildings = [{"folder": g["folder"], "config": Path(g["folder"]) / CONFIG_NAME}
                 for g in needs_run[n]]
    batch_dir = BATCH_BASE / f"{n}s" / "mdof"
    batch_dir.mkdir(parents=True, exist_ok=True)
    batch_dst = batch_dir / BATCH_NAME
    copy_batch_ida_buildings(
        cfg["templates"]["run_batch_ida_buildings"], batch_dst, buildings,
        window_name_index=2)
    print(f"wrote {batch_dst.relative_to(BATCH_BASE)}: {len(buildings)} building(s) to run "
          f"of {len(built[n])} group(s)")

wrote 3s\mdof\ida_femap695.py: 25 building(s) to run of 25 group(s)
wrote 5s\mdof\ida_femap695.py: 26 building(s) to run of 26 group(s)


## 6. Group → site map

One row per group, listing the member sites its IDA result stands for and whether that
result is already in place. Notebooks `053` / `060` / `061` use this to fan a group's
fragility back out to its sites — the IDA counterpart of the map `054` writes next to its
stripe selections.

In [12]:
rows = []
for n in STOREYS:
    for g in built[n]:
        sites = members[(n, g["group_id"])]
        rows.append({
            "storeys": n,
            "group_id": g["group_id"],
            "folder": Path(g["folder"]).as_posix(),
            "ida_tag": IDA_TAG,
            "representative_site": g["representative_site"],
            "representative_tag": g["representative_tag"],
            "n_sites": len(sites),
            "sites": " ".join(str(s) for s in sites),
            "results_migrated": (n, g["group_id"]) in migrated,
            "needs_run": g in needs_run[n],
        })

group_map = pd.DataFrame(rows)
map_path = MAP_DST / "group_folder_map.csv"
group_map.to_csv(map_path, index=False)
print(f"wrote {map_path}  ({len(group_map)} groups, "
      f"{int(group_map['needs_run'].sum())} still to run)")
group_map.head()

wrote C:\Users\clemettn\Documents\phd\results\06_group_ida_femap695\group_folder_map.csv  (51 groups, 51 still to run)


,storeys,group_id,folder,ida_tag,representative_site,representative_tag,n_sites,sites,results_migrated,needs_run
0,3,0,D:/08_wp1_fixed_record_sets/group_3s_00/3s/mdof,ida_femap695,0,3s_cbf_dc2_site0,8,0 4 8 19 25 26 27 29,False,True
1,3,1,D:/08_wp1_fixed_record_sets/group_3s_01/3s/mdof,ida_femap695,1,3s_cbf_dc2_site1,6,1 2 5 6 16 23,False,True
2,3,2,D:/08_wp1_fixed_record_sets/group_3s_02/3s/mdof,ida_femap695,3,3s_cbf_dc2_site3,1,3,False,True
3,3,3,D:/08_wp1_fixed_record_sets/group_3s_03/3s/mdof,ida_femap695,7,3s_cbf_dc2_site7,5,7 10 11 18 24,False,True
4,3,4,D:/08_wp1_fixed_record_sets/group_3s_04/3s/mdof,ida_femap695,9,3s_cbf_dc2_site9,3,9 20 22,False,True
